In [296]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "suda2006what")
starting_point = os.path.join(pathway, "original_data")

# complete_path_1 = os.path.join(original_data_pathway, "")


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [297]:
import pandas as pd
import numpy as np
import pyreadstat
import re


suda_2006 = [['FIN_Exp1_', '1' ], 
            ['FIN_Exp2_', '2'],
            ['FIN_Exp3_', '3']]

for  file_exp, exp_no  in suda_2006: 
    for dirpath, dirnames, filenames in os.walk(starting_point):
        for filename in [f for f in filenames if f.startswith(file_exp) and  f.endswith("sav")]:
            sav_filepath = os.path.join(dirpath, filename)
            ##get name of participant by capturing first group
            participant_name = re.match('.*_(.*).sav', filename).group(1)
            ##get new file names
            csv_filename = filename.replace(".sav", ".csv")
            csv_filepath = os.path.join(dirpath, csv_filename)
            ##write to csv
            read_file = pd.read_spss(sav_filepath, usecols=None, convert_categoricals=True)
            read_file = read_file.assign(participant=participant_name)
            read_file = read_file.assign(experiment=exp_no)
            read_file.to_csv(csv_filepath, encoding='utf-8-sig', index=False)


In [298]:
fulldf=[]
for dirpath, dirnames, filenames in os.walk(starting_point):
    for filename in [f for f in filenames if f.endswith(".csv")]:
        new_csv_filepath = os.path.join(dirpath, filename)
        # print(xlsx_filepath)
        fulldf.append(pd.read_csv(new_csv_filepath))

In [299]:
for index, x in enumerate(fulldf):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x=x.rename(columns={"trial": "trial_wrong",
                        "session":"session_wrong"})
    # x['data_subset']="data_subset_" + str(index+1)
    x['study_id']="suda2006what"
    fulldf[index]=x
fulldf = pd.concat(fulldf, ignore_index=True, sort=False)

In [300]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

In [301]:
fulldf.rename(columns={"tri": "trial",
    "ses":"session"}, inplace=True)

complete_path_age = os.path.join(starting_point, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')
fulldf.rename(columns={"age_y": "age_in_years"}, inplace=True) 

fulldf.rename(columns={
    # 'type':"trial_type", 
                       'pb1':"pointed_at_both_1st_choice", 
                       'cc1': "changed_in_1st_choice", 
                       'pb2':"pointed_at_both_2nd_choice", 
                       'cc2':"changed_in_2nd_choice", 
                       'fin1':"hesitated_in_1st_choice", 
                       'fin2':"hesitated_in_2nd_choice",
                        'lclt':"lc_or_lt_trial",  
                        'am2':"amount_of_chosen_liquid_2nd_choice", 
                        'ldlt':"ld_or_lt_trial",
                        #   'lk2', 
                          'lslm':"ls_or_lm_trial"}, inplace=True)

fulldf.columns

yes_no_list = ['pointed_at_both_1st_choice',
       'changed_in_1st_choice', 'pointed_at_both_2nd_choice',
       'changed_in_2nd_choice', 'hesitated_in_1st_choice',
       'hesitated_in_2nd_choice']
for x in yes_no_list:
    fulldf[x].replace(1, "true", inplace=True, regex=True)
    fulldf[x].replace(0, "false", inplace=True, regex=True)


trial_subtypes = [['lc_or_lt_trial', 1, 'lc_trial'],
        ['lc_or_lt_trial', 2,'lt_trial'],
        ['ld_or_lt_trial',1,'ld_trial'],
        ['ld_or_lt_trial', 2, 'lt_trial'],
        ['ls_or_lm_trial',1, 'ls_trial'],
        ['ls_or_lm_trial',2, 'lm_trial']]

for x,y,k in trial_subtypes:
    fulldf[x].replace(y, k, inplace=True, regex=True)

In [302]:

trial_dict = {1:{11:'clear_same',
                12:'clear_different',
                21:'opaque_same',
                22:'opaque_different',
                0:'no_testing_trial'},
             2:{11:'transfer_same',
                21:'transfer_different',
                20:'no_transfer_different',
                10:'no_transfer_same',
                0:'no_testing_trial'},
             3:{0:'no_testing_trial',
                4:'4-cup',
                8:'8-cup'}}
new_column = []
for index, row in fulldf.iterrows():
    if row['experiment'] in trial_dict.keys():
        #check if the trial type is in our dictionary
        if row['type'] in trial_dict[row['experiment']].keys():
            new_value = trial_dict[row['experiment']][row['type']]
            new_column.append(new_value)
        else:
            #print(f'could not find trial type for {row}')
            new_column.append(np.nan)
    else:
        #print(f'could not find experiment for {row}')
        new_column.append(np.nan)
        pass
fulldf = fulldf.assign(trial_type = new_column)


In [303]:
complete_path_1 = os.path.join(starting_point, "suda2004piagetian_exp1_dates.csv")
complete_path_2 = os.path.join(starting_point, "suda2004piagetian_exp2_dates.csv")
complete_path_3 = os.path.join(starting_point, "suda2004piagetian_exp3_dates.csv")

date_1 = pd.read_csv(complete_path_1)
date_2 = pd.read_csv(complete_path_2)
date_3 = pd.read_csv(complete_path_3)

data_frames=[date_1, date_2, date_3]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"experiment": "experiment_temp",
                      "study_id":"study_id_temp",
                    #   'year':"year_temp",
                    #   "month":"month_temp",
                    #   "day":"day_temp",
                      "participant":"participant_temp",
                      "session":"session_temp",
                      "trial":"trial_temp"}, inplace=True)
    data_frames[index]=x
new_df=data_frames[0]
date_full = pd.concat(data_frames, ignore_index=True, sort=False)
date_full.columns

Index(['study_id_temp', 'experiment_temp', 'year', 'month', 'day',
       'participant_temp', 'session_temp', 'trial_temp'],
      dtype='object')

In [304]:

date_full=date_full.sort_values(by = ['experiment_temp','participant_temp'])
date_full=date_full.values.tolist()
fulldf=fulldf.sort_values(by = ['experiment','participant'])


fulldf = fulldf[['study_id','experiment', 'participant', 'age_in_years','sex', 
                 'species','session', 'trial', 
        'trial_type','lc_or_lt_trial', 'ld_or_lt_trial','ls_or_lm_trial', 
        'pointed_at_both_1st_choice',
       'changed_in_1st_choice', 'pointed_at_both_2nd_choice',
       'changed_in_2nd_choice', 'hesitated_in_1st_choice',
       'hesitated_in_2nd_choice',  'amount_of_chosen_liquid_2nd_choice', 
    #    'lk2'
       ]]
fulldf.dropna(subset=['trial'], inplace=True)
fulldf=fulldf.values.tolist()


In [305]:
print(len(fulldf))
print(len(date_full))

4020
4020


In [306]:
combined_lol = [lol_1+lol_2 for lol_1,lol_2 in zip(fulldf,date_full)]

fulldf_new = pd.DataFrame(combined_lol, columns=['study_id','experiment', 'participant', 'age_in_years','sex', 
                 'species','session', 'trial', 
        'trial_type','lc_or_lt_trial', 'ld_or_lt_trial','ls_or_lm_trial', 
        'pointed_at_both_1st_choice',
       'changed_in_1st_choice', 'pointed_at_both_2nd_choice',
       'changed_in_2nd_choice', 'hesitated_in_1st_choice',
       'hesitated_in_2nd_choice',  'amount_of_chosen_liquid_2nd_choice',
       'study_id_temp', 'experiment_temp', 'year', 'month', 'day',
       'participant_temp', 'session_temp', 'trial_temp'])

In [307]:
# fulldf.columns

fulldf_new = fulldf_new[['study_id','experiment', 
                        #  'experiment_temp',
                         'year','month','day', 'participant',
                        #  'participant_temp', 
                         'age_in_years','sex', 
                 'species','session',
               #   'session_temp', 
                 'trial', 
               #   "trial_temp",
        'trial_type','lc_or_lt_trial', 'ld_or_lt_trial','ls_or_lm_trial', 
        'pointed_at_both_1st_choice',
       'changed_in_1st_choice', 'pointed_at_both_2nd_choice',
       'changed_in_2nd_choice', 'hesitated_in_1st_choice',
       'hesitated_in_2nd_choice',  'amount_of_chosen_liquid_2nd_choice', 
    #    'lk2'
       ]]

In [308]:
for index in range(1,4):
    exp = fulldf_new[fulldf_new['experiment'] == index]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'suda2006what_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'suda2006what_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)